# Train 6 Model Multi-Horizon XGBoost (Delta-based)

Setiap model memprediksi **perubahan** (delta) PM2.5 dari nilai saat ini,
bukan nilai absolut. Hasil: prediksi yang natural dan tidak meledak.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from supabase import create_client, Client
import os
from datetime import datetime, timedelta
import joblib

In [2]:
SUPABASE_URL = os.getenv("SUPABASE_URL", "")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

if not SUPABASE_URL or not SUPABASE_KEY:
    env_path = "../.env"
    if os.path.exists(env_path):
        for line in open(env_path):
            if "=" in line and not line.startswith("#"):
                k, _, v = line.partition("=")
                os.environ[k.strip()] = v.strip().strip('"')
    SUPABASE_URL = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f"Connected")

Connected


### 1. Ambil Data

In [3]:
TABLE = "tb_konsentrasi_gas"
days_back = 14
since = (datetime.now() - timedelta(days=days_back)).isoformat()

all_data = []
offset = 0
while True:
    resp = supabase.table(TABLE) \
        .select("pm25_ugm3,pm10_ugm3,no2_ugm3,co_ugm3,temperature,humidity,created_at") \
        .gte("created_at", since) \
        .order("created_at", desc=False) \
        .range(offset, offset + 999) \
        .execute()
    batch = resp.data
    if not batch:
        break
    all_data.extend(batch)
    offset += len(batch)

df = pd.DataFrame(all_data)
for col in ["pm25_ugm3","pm10_ugm3","no2_ugm3","co_ugm3","temperature","humidity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
df = df.dropna(subset=["pm25_ugm3","pm10_ugm3","no2_ugm3","co_ugm3","temperature","humidity"])
df = df.set_index("created_at").sort_index()
df = df[~df.index.duplicated(keep="first")]
df.rename(columns={"pm25_ugm3": "pm2.5"}, inplace=True)

print(f"Total data: {len(df)} baris")

Total data: 11762 baris


### 2. Feature Engineering

In [4]:
RAW_COLS = ["pm2.5", "pm10_ugm3", "co_ugm3", "no2_ugm3", "temperature", "humidity"]

feat = df.copy()

for col in RAW_COLS:
    feat[f"{col}_lag_1min"] = feat[col].shift(1)
    feat[f"{col}_lag_5min"] = feat[col].shift(5)
    feat[f"{col}_lag_15min"] = feat[col].shift(15)
    feat[f"{col}_lag_60min"] = feat[col].shift(60)
    feat[f"{col}_rolling_mean_5min"] = feat[col].rolling(5).mean()
    feat[f"{col}_rolling_std_5min"] = feat[col].rolling(5).std()
    feat[f"{col}_rolling_mean_15min"] = feat[col].rolling(15).mean()

feat["minute"] = feat.index.minute
feat["hour"] = feat.index.hour
feat["dayofweek"] = feat.index.dayofweek

print(f"Fitur base: {len(feat.columns)} kolom, {len(feat)} baris")

Fitur base: 51 kolom, 11762 baris


### 3. Train 6 Model — Target = DELTA (perubahan dari nilai sekarang)

In [5]:
HORIZONS = [5, 10, 15, 20, 25, 30]
EXCLUDE_FEATURES = ["minute", "hour", "dayofweek"]
FEATURE_COLS = [c for c in feat.columns if c not in ["target_now", "delta"] + EXCLUDE_FEATURES]
print(f"Fitur dipakai: {len(FEATURE_COLS)} (exclude: {EXCLUDE_FEATURES})")

models = {}
metrics = {}

for h in HORIZONS:
    print(f"\n--- Training horizon {h} min ---")

    df_h = feat.copy()
    future_val = df_h["pm2.5"].shift(-h)
    current_val = df_h["pm2.5"]
    df_h["delta"] = future_val - current_val
    df_h["target_now"] = current_val
    df_h = df_h.dropna()

    split_idx = int(len(df_h) * 0.8)
    X_tr = df_h.iloc[:split_idx][FEATURE_COLS]
    y_tr = df_h.iloc[:split_idx]["delta"]
    X_te = df_h.iloc[split_idx:][FEATURE_COLS]
    y_te = df_h.iloc[split_idx:]["delta"]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror",
        verbosity=0,
    )
    model.fit(X_tr, y_tr)

    pred_delta = model.predict(X_te)
    mae_delta = mean_absolute_error(y_te, pred_delta)

    actual_future = df_h.iloc[split_idx:]["target_now"] + y_te
    pred_future = df_h.iloc[split_idx:]["target_now"] + pred_delta
    mae_actual = mean_absolute_error(actual_future, pred_future)

    models[h] = model
    metrics[h] = {"mae_delta": round(mae_delta, 2), "mae_actual": round(mae_actual, 2)}
    print(f"  MAE delta: {mae_delta:.2f}, MAE aktual: {mae_actual:.2f}")

print("\n=== Ringkasan ===")
for h, m in metrics.items():
    print(f"  H={h:2d}min | MAE delta={m['mae_delta']:.2f} | MAE aktual={m['mae_actual']:.2f}")

Fitur dipakai: 48 (exclude: ['minute', 'hour', 'dayofweek'])

--- Training horizon 5 min ---
  MAE delta: 18.51, MAE aktual: 18.51

--- Training horizon 10 min ---
  MAE delta: 20.29, MAE aktual: 20.29

--- Training horizon 15 min ---
  MAE delta: 24.68, MAE aktual: 24.68

--- Training horizon 20 min ---
  MAE delta: 29.07, MAE aktual: 29.07

--- Training horizon 25 min ---
  MAE delta: 31.51, MAE aktual: 31.51

--- Training horizon 30 min ---
  MAE delta: 31.39, MAE aktual: 31.39

=== Ringkasan ===
  H= 5min | MAE delta=18.51 | MAE aktual=18.51
  H=10min | MAE delta=20.29 | MAE aktual=20.29
  H=15min | MAE delta=24.68 | MAE aktual=24.68
  H=20min | MAE delta=29.07 | MAE aktual=29.07
  H=25min | MAE delta=31.51 | MAE aktual=31.51
  H=30min | MAE delta=31.39 | MAE aktual=31.39


### 4. Simpan Model

In [6]:
joblib.dump(models, "xgb_pm25_multistep.pkl")
print("6 model tersimpan ke: xgb_pm25_multistep.pkl")
print(f"Horizons: {list(models.keys())}")
print(f"Feature cols: {len(models[5].feature_names_in_)}")

6 model tersimpan ke: xgb_pm25_multistep.pkl
Horizons: [5, 10, 15, 20, 25, 30]
Feature cols: 48


### 5. Quick Test

In [7]:
feat_test = feat.dropna().iloc[[-1]]
X_row = feat_test[FEATURE_COLS]

current = float(feat_test["pm2.5"].values[0])
print(f"PM2.5 saat ini: {current:.1f} µg/m³\n")
print("Prediksi (nilai absolut):")
for h in HORIZONS:
    delta = float(models[h].predict(X_row)[0])
    predicted = current + delta
    print(f"  +{h:2d} min → delta={delta:+.1f} → {predicted:.1f} µg/m³")

PM2.5 saat ini: 20.0 µg/m³

Prediksi (nilai absolut):
  + 5 min → delta=+17.1 → 37.1 µg/m³
  +10 min → delta=+21.3 → 41.3 µg/m³
  +15 min → delta=+21.9 → 41.9 µg/m³
  +20 min → delta=+50.4 → 70.4 µg/m³
  +25 min → delta=+56.3 → 76.3 µg/m³
  +30 min → delta=+47.6 → 67.6 µg/m³
